# 00 - Orquestador del pipeline

## Como se activa el flujo -- arquitectura en dos etapas

Este pipeline no tiene un solo "botón de encendido" que corra las 6 notebooks de punta a punta y eso es una decisión tomada a conciencia. Lo separé en dos etapas según qué tan caro es recalcular cada cosa:

**Etapa 1 -- Transformación pesada (manual / bajo demanda):**
`01_ingesta_datos_crudos_s3` -> `02_perfilamiento_esquemas` -> `03_etl_limpieza_spark` ->`04_joins_agregaciones_spark`. Son los pasos que usa Spark para pasar de crudo (Bronze) a limpio (Silver) y de limpio a agregados (Gold). Los ejecuto manualmente cuando llegan datos nuevos de la TLC, porque:
1. El crudo no cambia todos los días; no tiene sentido recalcular Gold en cada ejecución si Silver no cambió.
2. Son procesos pesados de Spark (JVM, JARs de S3A, shuffles) que en una instancia t2.medium (~4GB RAM) compite por memoria con cualquier otra cosa corriendo al mismo tiempo. Automatizar su re-ejecución repetida en esta instancia específica resultó poco confiable (cuota de disco, memoria), así que preferí ejecutarlos deliberadamente y dejar que ESTE orquestador se enfoque en lo que sí es seguro automatizar.

**Etapa 2 -- sincronización ligera (automatizada aquí, en este notebook):**
`06_sync_metadatos_rds` -> `05_export_rds`. Ninguno de los dos usa Spark; ambos son idempotentes (pueden correr las veces que sea sin acumular datos raros) y ambos leen datos que ya están calculados en S3, solo los sincronizan hacia RDS. Esto es lo que sí tiene sentido re-disparar seguido (por ejemplo, antes de una demo, o en un schedule), y es justo lo que hace este notebook.

## Manejo de credenciales
Mismo patron que en el resto del proyecto: si `MYSQL_HOST` / `MYSQL_USER` / `MYSQL_DB` ya
estan como variables de entorno las uso directo: si no, las pido en el momento. La
contrasena siempre se pide con `getpass` y ademas la exporto a `os.environ` para que los
notebooks que este script lanza como subprocesos (`06` y `05`) la hereden sin tener que
volver a pedirla -- un subproceso sin terminal interactiva se quedaria colgado esperando
un `input()`/`getpass()` que nunca llega.

In [1]:
import os
import getpass
import subprocess
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import create_engine, text

NOTEBOOKS_DIR = "."  

# Etapa 2 unicamente -- ver explicacion arriba. 03 y 04 se corren manualmente aparte.
secuencia = [
    "06_sync_metadatos_rds.ipynb",
    "05_export_rds.ipynb",
]

mysql_host = os.getenv("MYSQL_HOST") or input("Endpoint de RDS MySQL: ")
mysql_port = int(os.getenv("MYSQL_PORT", "3306"))
mysql_user = os.getenv("MYSQL_USER") or input("Usuario de MySQL: ")
mysql_db = os.getenv("MYSQL_DB") or input("Base de datos (schema): ")
mysql_password = os.getenv("MYSQL_PASSWORD") or getpass.getpass("Contrasena de MySQL: ")

os.environ["MYSQL_HOST"] = mysql_host
os.environ["MYSQL_PORT"] = str(mysql_port)
os.environ["MYSQL_USER"] = mysql_user
os.environ["MYSQL_DB"] = mysql_db
os.environ["MYSQL_PASSWORD"] = mysql_password

connection_string = f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_db}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Conexion a RDS exitosa, base de datos:", mysql_db)

Endpoint de RDS MySQL:  database-1.cd8w4cuu6a79.us-west-1.rds.amazonaws.com
Usuario de MySQL:  admin
Base de datos (schema):  proyecto_integrador_alan
Contrasena de MySQL:  ········


Conexion a RDS exitosa, base de datos: proyecto_integrador_alan


## Ejecucion secuencial
Cada notebook corre en su propio kernel independiente (via `jupyter nbconvert --execute`),
asi que un fallo en uno no deja procesos zombie que afecten al siguiente. Si un paso
falla, se detiene la secuencia y se marcan como `SKIPPED` los que faltan -- no tiene caso
exportar a RDS si la sincronizacion de metadatos no se completo.

In [2]:
resultados = []
detener = False

for nombre_notebook in secuencia:
    ruta = os.path.join(NOTEBOOKS_DIR, nombre_notebook)

    if detener:
        print(f"Se omite {nombre_notebook} porque un paso anterior fallo.")
        resultados.append({
            "notebook_name": nombre_notebook,
            "status": "SKIPPED",
            "started_at": datetime.now(timezone.utc).replace(tzinfo=None),
            "finished_at": datetime.now(timezone.utc).replace(tzinfo=None),
            "error_message": "Omitido: un paso anterior de la secuencia fallo.",
        })
        continue

    inicio = datetime.now(timezone.utc)
    print(f"\n=== Ejecutando {nombre_notebook} ===")

    proceso = subprocess.run(
        [
            "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--inplace",
            "--ExecutePreprocessor.timeout=-1",
            ruta,
        ],
        capture_output=True,
        text=True,
    )

    fin = datetime.now(timezone.utc)
    exito = proceso.returncode == 0
    estado = "SUCCESS" if exito else "FAILED"
    mensaje_error = None if exito else (proceso.stderr[:2000] + "\n...(recortado)...\n" + proceso.stderr[-4000:])

    print(f"Estado: {estado}  (duracion: {fin - inicio})")
    if not exito:
        print("Detalle del error:")
        print(mensaje_error)
        detener = True

    resultados.append({
        "notebook_name": nombre_notebook,
        "status": estado,
        "started_at": inicio.replace(tzinfo=None),
        "finished_at": fin.replace(tzinfo=None),
        "error_message": mensaje_error,
    })


=== Ejecutando 06_sync_metadatos_rds.ipynb ===
Estado: SUCCESS  (duracion: 0:00:08.811462)

=== Ejecutando 05_export_rds.ipynb ===
Estado: SUCCESS  (duracion: 0:01:46.131063)


## Registro de la ejecución en `pipeline_runs`
Guardo cada ejecución (exitosa o no) en RDS -- es lo que alimenta la pestaña "Salud del pipeline" del dashboard de Streamlit, y es la evidencia de que el pipeline se monitorea a sí mismo, no solo se ejecuta.

In [4]:
insert_sql = text("""
    INSERT INTO pipeline_runs (notebook_name, status, started_at, finished_at, error_message)
    VALUES (:notebook_name, :status, :started_at, :finished_at, :error_message)
""")

with engine.begin() as conn:
    for resultado in resultados:
        conn.execute(insert_sql, resultado)

print(f"{len(resultados)} ejecuciones registradas en pipeline_runs")

2 ejecuciones registradas en pipeline_runs


In [6]:
ultimas_ejecuciones = pd.read_sql(
    "SELECT * FROM pipeline_runs ORDER BY run_id DESC LIMIT 10",
    engine,
)
ultimas_ejecuciones

,run_id,notebook_name,status,started_at,finished_at,error_message
0,4,05_export_rds.ipynb,SUCCESS,2026-09-19 06:38:43,2026-09-19 06:40:29,None
1,3,06_sync_metadatos_rds.ipynb,SUCCESS,2026-09-19 06:38:34,2026-09-19 06:38:43,None
2,2,05_export_rds.ipynb,SUCCESS,2026-09-19 06:38:43,2026-09-19 06:40:29,None
3,1,06_sync_metadatos_rds.ipynb,SUCCESS,2026-09-19 06:38:34,2026-09-19 06:38:43,None


In [7]:
engine.dispose()
print("Conexion a RDS cerrada")

Conexion a RDS cerrada
